# CERDOS 54 · Presupuesto computacional igualado en GPU L4
### Experimento complementario · FaroPigSeg (versión 3: semilla automática; basta con «Ejecutar todas» en cada sesión)

**Objetivo:** comparar **YOLO11n-seg, YOLO26n-seg y Mask R-CNN ResNet50-FPN** bajo un **límite común de tiempo de entrenamiento en la misma GPU** y con **tres semillas (2026, 2027, 2028)**. Es una comparación de **recetas de entrenamiento bajo restricción de tiempo**, NO un aislamiento causal perfecto del efecto de arquitectura ni una nueva validación externa. No se modifican ni las particiones ni los nueve entrenamientos originales (NB22, NB31 y NB32).

**Orden de ejecución**
1. En Colab: **Entorno de ejecución → Cambiar tipo → GPU L4**.
2. Ejecutar el notebook con `RUN_PILOT=True` y `RUN_FULL=False` (predeterminado). El piloto usa solo train/val y registra tiempos observados.
3. Ejecutar la **celda 5b** (precarga de train/val y re-medición de Mask R-CNN sin la lectura lenta de Drive) y anotar el presupuesto recomendado. Revisar también el consumo de unidades. Editar `BUDGET_MINUTES_PER_RUN`, `RUN_PILOT=False`, `RUN_FULL=True` y `ACTIVE_SEEDS=(2026,)` (luego 2027 y 2028) en la configuración, volver a ejecutar.
4. Tras el entrenamiento, revisar `matched_budget_summary.csv`, hashes, tiempos y curvas de validación. **Este notebook NO evalúa el test**. Las predicciones sobre los 373 casos del test se analizarán posteriormente con un evaluador común, etiquetadas como retrospectivas (test consultado anteriormente).

**Advertencias:** 130 unidades de Colab **no equivalen a un número fijo de horas L4**; Colab determina el consumo dinámicamente. El límite es aproximado por épocas (incluye evaluación/guardado) y se registrará el tiempo efectivamente observado. Si una corrida supera el presupuesto, NO se declarará equivalente sin revisar tolerancias. La comparación conserva recetas distintas (YOLO `optimizer=auto`, Mask R-CNN AdamW y entradas diferentes), que deberán declararse como factores de confusión.


## 1. Configuración controlada (editar solo esta celda para elegir piloto o entrenamiento)

In [ ]:
# SESIONES DE ENTRENAMIENTO COMPLETO (versión 3). No hay que editar nada: usar «Ejecutar todas» en cada sesión.
# La semilla se elige sola: la primera de SEEDS cuyas tres corridas (YOLO11n, YOLO26n, Mask R-CNN) no tienen finish.json.
RUN_PILOT = False
RUN_FULL = True
RUN_REMEASURE = False  # la re-medición ya está guardada; la celda 5b solo hará la precarga de train/val
MASK_EPOCHS_REFERENCE = 12  # épocas de Mask R-CNN en la receta original (NB06/NB32), para comparar el presupuesto
BUDGET_MINUTES_PER_RUN = 60.0  # FIJO para las 9 corridas. No cambiar.
SEEDS = (2026, 2027, 2028)
ACTIVE_SEEDS = 'auto'  # se resuelve en la sección 7 a partir de los finish.json existentes
GPU_NAME_REQUIRED = 'L4'  # No cambia el dataset; detiene la ejecución si no corresponde
RELATIVE_WALL_TIME_TOLERANCE = 0.10  # tolerancia descriptiva máxima de 10% respecto al objetivo
PROTOCOL_ID = 'CERDOS54_L4_EQUAL_TIME_v1'
YOLO_EPOCH_CAP = 250  # time= tiene prioridad sobre epochs en Ultralytics
MASK_EPOCH_CAP = 250
MASK_BATCH = 1
YOLO_BATCH = 8
YOLO_IMGSZ = 640
PILOT_MASK_TRAIN_IMAGES = 96  # piloto parcial, NO se usa para estimar rendimiento científico
PILOT_YOLO_FRACTION = 0.12
PILOT_YOLO_EPOCHS = 2
assert BUDGET_MINUTES_PER_RUN == 60.0, 'El presupuesto debe ser idéntico en las 9 corridas.'
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
print('Etapa: entrenamiento completo | semilla: automática (primera pendiente) | presupuesto:', BUDGET_MINUTES_PER_RUN, 'min por corrida')
print('Duración de una sesión: ≈3 h 10 min (3 modelos × 60 min + preparación y precarga)')


## 2. Entorno Colab, librerías y Drive

In [ ]:
import importlib.util, subprocess, sys
required = {'ultralytics':'ultralytics', 'pycocotools':'pycocotools', 'yaml':'pyyaml', 'cv2':'opencv-python-headless', 'pandas':'pandas', 'PIL':'Pillow'}
missing=[pip for mod,pip in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import os, json, yaml, time, hashlib, platform, random, gc, shutil, io
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch, torchvision, cv2, matplotlib.pyplot as plt
from ultralytics import YOLO
import ultralytics
from google.colab import drive
assert torch.cuda.is_available(), 'Activar GPU en Colab antes de continuar.'
GPU = torch.cuda.get_device_name(0)
print('GPU:',GPU,'| Torch:',torch.__version__,'| Torchvision:',torchvision.__version__,'| Ultralytics:',ultralytics.__version__)
assert GPU_NAME_REQUIRED.lower() in GPU.lower(), f'Esta comparación estaba diseñada para L4: {GPU}'
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Carpeta original no localizada: {ROOT}'
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
YAML=SPLIT/'data_fecha.yaml'; MANIFEST=SPLIT/'particion_fecha_manifest.json'
OUT=ROOT/'03_Resultados'/'54_Presupuesto_Equiparado_L4'
OUT.mkdir(parents=True,exist_ok=True)
assert all(x.is_file() for x in (YAML,MANIFEST)), 'Faltan los manifiestos congelados; no reconstruir particiones.'
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def write_json_atomic(path, payload):
    p=Path(path); tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(payload,indent=2,ensure_ascii=False,default=str),encoding='utf-8');os.replace(tmp,p)
print('Salida nueva:',OUT)


## 3. Auditoría de particiones congeladas (sin evaluar el test)

In [ ]:
# SHA256 fijado por el experimento original NB32. FAIL-CLOSED ante cualquier variación.
EXPECTED_YAML_SHA='9a223b47038c661303c1d12a35cbdcb212ed51529c8c773c4c9c66998706d1d9'
assert sha256_file(YAML)==EXPECTED_YAML_SHA, 'El YAML difiere del estudio original: ABORTAR.'
cf=yaml.safe_load(YAML.read_text(encoding='utf-8'))
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
assert set(manifest['asignaciones_fecha'].values())=={'train','val','test'}
parts={}; dates={}
for s,n_expected,d_expected in [('train',949,16),('val',196,4),('test',373,5)]:
    p=Path(cf[s]); assert p.is_file(),f'Falta lista {s}: {p}'
    parts[s]=[Path(x.strip()) for x in p.read_text(encoding='utf-8').splitlines() if x.strip()]
    dates[s]={p.name[:10] for p in parts[s]}
    assert len(parts[s])==n_expected and len(dates[s])==d_expected,(s,len(parts[s]),len(dates[s]))
    assert len({str(p) for p in parts[s]})==len(parts[s])
    assert all(manifest['asignaciones_fecha'].get(d)==s for d in dates[s]), 'Fechas contra manifiesto no coinciden'
    print(s,':',len(parts[s]),'imágenes,',len(dates[s]),'fechas')
for a,b in [('train','val'),('train','test'),('val','test')]:
    assert not ({str(p) for p in parts[a]}&{str(p) for p in parts[b]})
    assert not(dates[a]&dates[b])
# Sólo train y val se inspeccionan o cargan; el test se enumera para auditoría de IDs/fechas.
for s in ('train','val'):
    for p in parts[s]:
        assert p.is_file(), f'Imagen ausente: {p}'
        label=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        assert label.is_file(),f'Etiqueta ausente: {label}'
assert cf.get('names') in ({0:'cerdo'},{0:'pig'},['cerdo'],['pig'])
HASHES={'data_fecha_yaml':sha256_file(YAML),'date_manifest':sha256_file(MANIFEST),
        **{s+'_txt':sha256_file(Path(cf[s])) for s in ('train','val','test')}}
PROTOCOL={"protocol_id":PROTOCOL_ID,"objective":"equal L4 elapsed GPU-session wall time, not equal FLOPs or matched architecture hyperparameters",
    "split_counts":{s:len(parts[s]) for s in parts},"split_dates":{s:len(dates[s]) for s in dates},
    "seeds":list(SEEDS),"gpu":GPU,"budgets_min":BUDGET_MINUTES_PER_RUN,
    "frozen_inputs":HASHES,"torch":torch.__version__,"torchvision":torchvision.__version__,
    "ultralytics":ultralytics.__version__,"mask_batch":MASK_BATCH,"yolo_batch":YOLO_BATCH,
    "test_used_for_training_or_tuning":False,
    "interpretation":"retrospective sensitivity; test was inspected in earlier experiments of this project"}
protocol_file=OUT/'protocol_config_v1.json'
if protocol_file.exists():
    prev=json.loads(protocol_file.read_text(encoding='utf-8'))
    # Parametrización del piloto independiente del entrenamiento; para cambiar minutos crear carpeta de experimento distinta.
    if prev.get('frozen_inputs')!=HASHES: raise RuntimeError('Las fuentes congeladas cambiaron: ABORTAR.')
    if prev.get('budgets_min')!=PROTOCOL['budgets_min']:
        existing=list((OUT/'matched').rglob('finish.json')) if (OUT/'matched').exists() else []
        existing+=list((OUT/'matched').rglob('last.pt')) if (OUT/'matched').exists() else []
        existing+=list((OUT/'matched').rglob('maskrcnn_last.pt')) if (OUT/'matched').exists() else []
        if existing: raise RuntimeError('Presupuesto modificado tras iniciar entrenamiento. Usar otra carpeta de experimento.')
        write_json_atomic(protocol_file,PROTOCOL)
else:write_json_atomic(protocol_file,PROTOCOL)
print('✅ Auditoría: train/val/test disjuntos, sin test utilizado en el ajuste. SHA YAML:',HASHES['data_fecha_yaml'])


## 4. Conjunto Mask R-CNN y evaluador COCO de validación de NB06/NB32

Esta celda conserva el cargador de polígonos YOLO, la inicialización COCO de Mask R-CNN, el congelamiento de capas y el evaluador de máscaras **de NB32**. Se utiliza únicamente la partición de **validación**, con 196 imágenes. Las rutinas de evaluación se aplicarán bajo el mismo protocolo a nuevos checkpoints; los resultados de test se calcularán después, en un experimento separado.

In [ ]:
import numpy as np, pandas as pd,torch,cv2,yaml,random,math
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision.transforms.functional import to_tensor

class PigMasks(Dataset):
    def __init__(self,imagelist):
        self.paths=[Path(s.strip()) for s in Path(imagelist).read_text(encoding='utf8').splitlines() if s.strip()]
        if not self.paths:raise RuntimeError('La partición está vacía: '+str(imagelist))
        missing=[str(p) for p in self.paths if not p.is_file()]
        if missing: raise FileNotFoundError('Hay imágenes ausentes: '+str(missing[:3]))
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        p=self.paths[index]
        rgb=np.asarray(Image.open(p).convert('RGB'))
        h,w=rgb.shape[:2]
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lab.is_file():raise FileNotFoundError('Etiqueta no localizada: '+str(lab))
        masks=[];boxes=[]
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip(): continue
            vals=row.split()
            if int(vals[0])!=0:raise ValueError('Clase inesperada en '+str(lab))
            points=np.asarray([float(v) for v in vals[1:]],dtype=np.float32)
            if len(points)<6 or len(points)%2:raise ValueError('Polígono inválido en '+str(lab))
            if not np.isfinite(points).all() or np.any((points<0)|(points>1)):
                raise ValueError('Coordenadas no normalizadas en '+str(lab))
            coords=points.reshape(-1,2)*np.asarray([w,h],dtype=np.float32)
            xy=np.rint(coords).astype(np.int32)
            xy[:,0]=np.clip(xy[:,0],0,w-1);xy[:,1]=np.clip(xy[:,1],0,h-1)
            mask=np.zeros((h,w),np.uint8)
            cv2.fillPoly(mask,[xy],1)
            yy,xx=np.where(mask)
            if len(xx)<3: continue
            x0,x1=int(xx.min()),int(xx.max()+1);y0,y1=int(yy.min()),int(yy.max()+1)
            if x1<=x0 or y1<=y0:continue
            masks.append(mask);boxes.append([x0,y0,x1,y1])
        if not masks:raise RuntimeError('Imagen sin máscaras válidas: '+p.name)
        tar={'boxes':torch.as_tensor(boxes,dtype=torch.float32),
             'labels':torch.ones((len(masks),),dtype=torch.int64),
             'masks':torch.as_tensor(np.stack(masks),dtype=torch.uint8),
             'image_id':torch.tensor([index],dtype=torch.int64),
             'area':torch.as_tensor([int(m.sum()) for m in masks],dtype=torch.float32),
             'iscrowd':torch.zeros((len(masks),),dtype=torch.int64)}
        return to_tensor(rgb.copy()),tar

def collate_fn(batch):return tuple(zip(*batch))

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_maskrcnn(pretrained=True):
    model=maskrcnn_resnet50_fpn(weights='DEFAULT' if pretrained else None,
                               weights_backbone=None,min_size=512,max_size=768,
                               trainable_backbone_layers=3)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,2)
    in_channels=model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor=MaskRCNNPredictor(in_channels,256,2)
    # Congelar los mismos niveles, tanto en arranque con pesos COCO como en reanudación sin descarga.
    for name,parameter in model.backbone.body.named_parameters():
        if name.startswith(('conv1.', 'bn1.', 'layer1.')):
            parameter.requires_grad_(False)
    model.roi_heads.score_thresh=0.001
    model.roi_heads.detections_per_img=100
    return model

def config_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils
from contextlib import redirect_stdout
import io

def encode_mask(arr):
    enc=maskutils.encode(np.asfortranarray(np.asarray(arr,dtype=np.uint8)))
    enc['counts']=enc['counts'].decode('ascii')
    return enc

def coco_metric_for_model(model,dataset,device,desc='Validación',include_rows=False):
    model.eval();gts=[];preds=[];image_records=[];annotations=[];aid=1
    for i in range(len(dataset)):
        image,target=dataset[i]
        height,width=image.shape[1:]
        gts.append({'id':i+1,'width':width,'height':height,'file_name':dataset.paths[i].name})
        gt_masks=target['masks'].numpy()
        for m in gt_masks:
            encoded=encode_mask(m)
            x,y,w,h=[float(v) for v in maskutils.toBbox({'size':encoded['size'],'counts':encoded['counts'].encode('ascii')})]
            annotations.append({'id':aid,'image_id':i+1,'category_id':1,'segmentation':encoded,
                                'bbox':[x,y,w,h],'area':int(m.sum()),'iscrowd':0});aid+=1
        with torch.no_grad():
            out=model([image.to(device)])[0]
        scores=out['scores'].detach().cpu().numpy()
        ms=(out['masks'].detach().cpu().numpy()[:,0]>.5) if len(scores) else np.zeros((0,height,width),dtype=bool)
        for j,score in enumerate(scores):
            if score<0.001: continue
            m=ms[j].astype(np.uint8)
            if not m.any():continue
            enc=encode_mask(m)
            preds.append({'image_id':i+1,'category_id':1,'segmentation':enc,'score':float(score)})
        if include_rows:
            filtered=np.where(scores>=.25)[0]
            predm=[ms[k].astype(bool) for k in filtered if k<len(ms)]
            p_scores=[float(scores[k]) for k in filtered if k<len(ms)]
            pairs=[]
            for gi,g in enumerate(gt_masks):
                for pj,pm in enumerate(predm):
                    inter=np.logical_and(g,pm).sum();union=np.logical_or(g,pm).sum()
                    iou=inter/max(1,union)
                    if iou>=.5:pairs.append((float(iou),gi,pj))
            used_gt=set();used_pred=set()
            for _,gi,pj in sorted(pairs,reverse=True):
                if gi not in used_gt and pj not in used_pred:
                    used_gt.add(gi);used_pred.add(pj)
            tp=len(used_gt)
            image_records.append({'imagen':dataset.paths[i].name,
                'fecha':dataset.paths[i].name[:10],
                'anotaciones_GT':len(gt_masks),'detecciones_conf_025':len(predm),
                'TP_IoU05':tp,'FP_IoU05':len(predm)-tp,'FN_IoU05':len(gt_masks)-tp,
                'confianza_media_pred':np.mean(p_scores) if p_scores else np.nan})
        if (i+1)%50==0:print(desc,i+1,'/',len(dataset))
    gt=COCO()
    gt.dataset={'images':gts,'annotations':annotations,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}
    with redirect_stdout(io.StringIO()):
        gt.createIndex()
        dt=gt.loadRes(preds) if preds else gt.loadRes([])
        ev=COCOeval(gt,dt,'segm');ev.evaluate();ev.accumulate();ev.summarize()
    return {'seg_map50_95':float(ev.stats[0]),'seg_map50':float(ev.stats[1]),
            'seg_map75':float(ev.stats[2]),'gt':len(annotations),
            'num_images':len(dataset),'pred_coarse':len(preds)},pd.DataFrame(image_records)


## 5. PILOTO: medir tiempo sin consumir el presupuesto de nueve entrenamientos

In [ ]:
# Experimento de perfil: las métricas de este piloto NO se publicarán como resultado principal.
# En YOLO se utiliza una fracción fija de train. En Mask R-CNN se eligen 96 IDs mediante RNG fijo.
# El tiempo del piloto NO predice perfectamente un entrenamiento completo (warmup, validación, caché, I/O).
from torch.utils.data import Subset

def pilot_yolo(model_name):
    outdir=OUT/'pilot'/model_name.replace('.pt','')
    marker=outdir/'pilot_metadata.json'
    if marker.exists():return json.loads(marker.read_text(encoding='utf-8'))
    outdir.mkdir(parents=True,exist_ok=True)
    torch.cuda.empty_cache()
    model=YOLO(model_name)
    t0=time.monotonic()
    model.train(data=str(YAML),epochs=PILOT_YOLO_EPOCHS,fraction=PILOT_YOLO_FRACTION,
                imgsz=YOLO_IMGSZ,batch=YOLO_BATCH,seed=2026,deterministic=True,
                device=0,workers=2,optimizer='auto',amp=True,patience=100,
                close_mosaic=0,plots=False,save=True,project=str(OUT/'pilot'),
                name=model_name.replace('.pt',''),exist_ok=True)
    torch.cuda.synchronize(); elapsed=time.monotonic()-t0
    info={'model':model_name,'pilot_seconds':elapsed,'pilot_train_fraction':PILOT_YOLO_FRACTION,
          'pilot_epochs':PILOT_YOLO_EPOCHS,'gpu':GPU,'excluded_from_final_analysis':True}
    write_json_atomic(marker,info)
    return info

def pilot_maskrcnn():
    outdir=OUT/'pilot'/'maskrcnn';outdir.mkdir(parents=True,exist_ok=True)
    marker=outdir/'pilot_metadata.json'
    if marker.exists():return json.loads(marker.read_text(encoding='utf-8'))
    torch.manual_seed(2026);np.random.seed(2026);random.seed(2026)
    ds=PigMasks(cf['train']);val=PigMasks(cf['val'])
    chosen=sorted(np.random.default_rng(2026).choice(len(ds),size=min(len(ds),PILOT_MASK_TRAIN_IMAGES),replace=False).tolist())
    loader=DataLoader(Subset(ds,chosen),batch_size=MASK_BATCH,shuffle=False,num_workers=0,collate_fn=collate_fn)
    model=build_maskrcnn(pretrained=True).to('cuda')
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    t0=time.monotonic();lossvals=[]
    model.train()
    for images,targets in loader:
        images=[x.cuda(non_blocking=True) for x in images]
        targets=[{k:v.cuda(non_blocking=True) for k,v in t.items()} for t in targets]
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):
            ld=model(images,targets);loss=sum(ld.values())
        assert torch.isfinite(loss), 'Pérdida MaskRCNN no finita en piloto.'
        scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
        scaler.step(opt);scaler.update();lossvals.append(float(loss.item()))
    torch.cuda.synchronize();train_seconds=time.monotonic()-t0
    t1=time.monotonic();metrics,_=coco_metric_for_model(model,val,torch.device('cuda'),desc='Piloto val')
    torch.cuda.synchronize();val_seconds=time.monotonic()-t1
    info={'model':'Mask R-CNN ResNet50-FPN','train_seconds_partial':train_seconds,
          'validation_seconds_all':val_seconds,'n_train_images':len(chosen),'n_val_images':len(val),
          'train_seconds_estimated_full_epoch':train_seconds*(len(ds)/len(chosen)),
          'pilot_train_loss_mean':np.mean(lossvals),'gpu':GPU,'excluded_from_final_analysis':True}
    write_json_atomic(marker,info)
    del model,opt,scaler,ds,val,loader;gc.collect();torch.cuda.empty_cache()
    return info

if RUN_PILOT:
    pilot_results=[]
    for kind in ('yolo11n-seg.pt','yolo26n-seg.pt'):
        print('PILOTO',kind)
        pilot_results.append(pilot_yolo(kind))
    print('PILOTO Mask R-CNN');pilot_results.append(pilot_maskrcnn())
    pd.DataFrame(pilot_results).to_csv(OUT/'pilot_summary.csv',index=False)
    print('✅ Piloto finalizado. Consulta:',OUT/'pilot_summary.csv')
else: print('Piloto omitido según configuración.')


## 5b. Precarga de train/val y re-medición de Mask R-CNN (versión 2)

El piloto original midió Mask R-CNN leyendo las imágenes directamente desde Google Drive (≈1,6 s/imagen), lo que sobrestima el tiempo de cómputo. Esta celda lee **una vez** todas las imágenes y etiquetas de **train y val** para dejarlas en la caché local de Colab, sin cambiar rutas, YAML ni hashes, y **sin tocar el test**. Después repite la medición de Mask R-CNN con las mismas 96 imágenes y la misma receta del piloto, y recomienda un presupuesto común.

**Ejecutar la precarga al inicio de cada sesión** (la caché se pierde al cerrar Colab). La re-medición solo hace falta una vez (`RUN_REMEASURE=True`); el resultado se guarda en `pilot_v2_remeasure.json` y no forma parte del análisis científico.

In [ ]:
# 5b. Precarga (Drive -> caché de Colab) y re-medición de Mask R-CNN. No modifica rutas, YAML, hashes ni el piloto. No toca el test.
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Subset

def _leer(p):
    p = Path(p)
    lab = Path(str(p).replace('/images/', '/labels/')).with_suffix('.txt')
    return len(p.read_bytes()) + len(lab.read_bytes())

t0 = time.monotonic()
for s in ('train', 'val'):
    with ThreadPoolExecutor(8) as ex:
        mb = sum(ex.map(_leer, parts[s])) / 1e6
    print(f'{s}: {len(parts[s])} imágenes precargadas ({mb:.0f} MB)')
print(f'Precarga completada en {time.monotonic()-t0:.0f} s')

remeasure_file = OUT / 'pilot_v2_remeasure.json'
if RUN_REMEASURE and not remeasure_file.exists():
    torch.manual_seed(2026); np.random.seed(2026); random.seed(2026)
    ds = PigMasks(cf['train'])
    idx = sorted(np.random.default_rng(2026).choice(len(ds), size=min(len(ds), PILOT_MASK_TRAIN_IMAGES), replace=False).tolist())
    loader = DataLoader(Subset(ds, idx), batch_size=MASK_BATCH, shuffle=False, num_workers=0, collate_fn=collate_fn)
    model = build_maskrcnn(pretrained=True).to('cuda'); model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    t0 = time.monotonic()
    for images, targets in loader:
        images = [x.cuda(non_blocking=True) for x in images]
        targets = [{k: v.cuda(non_blocking=True) for k, v in t.items()} for t in targets]
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            loss = sum(model(images, targets).values())
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update()
    torch.cuda.synchronize(); seconds = time.monotonic() - t0
    pilot = json.loads((OUT / 'pilot' / 'maskrcnn' / 'pilot_metadata.json').read_text(encoding='utf-8'))
    val_min = pilot['validation_seconds_all'] / 60          # validación completa (196 imágenes)
    epoch_min = seconds / len(idx) * len(ds) / 60 + val_min / 2   # se valida cada 2 épocas
    write_json_atomic(remeasure_file, {'seconds_per_image_cached': seconds / len(idx),
        'seconds_per_image_original_pilot': pilot['train_seconds_partial'] / pilot['n_train_images'],
        'estimated_minutes_per_epoch_incl_val': epoch_min, 'n_images': len(idx), 'gpu': GPU,
        'excluded_from_final_analysis': True, 'utc': datetime.now(timezone.utc).isoformat()})
    del model, opt, scaler, loader, ds; gc.collect(); torch.cuda.empty_cache()

if remeasure_file.exists():
    r = json.loads(remeasure_file.read_text(encoding='utf-8'))
    em = r['estimated_minutes_per_epoch_incl_val']
    print(f"\nMask R-CNN: {r['seconds_per_image_cached']:.2f} s/imagen con caché "
          f"(piloto original desde Drive: {r['seconds_per_image_original_pilot']:.2f} s/imagen)")
    print(f'≈ {em:.1f} min por época, incluida la validación')
    opciones = (45, 60, 75, 90, 105, 120)
    for b in opciones:
        print(f'  Presupuesto {b:>3} min/corrida -> ≈{int(b // em):>3} épocas de Mask R-CNN | '
              f'9 corridas ≈ {9*b/60:.1f} h de GPU (≈{3*b/60:.1f} h por sesión)')
    recomendado = next((b for b in opciones if b // em >= MASK_EPOCHS_REFERENCE), opciones[-1])
    print(f'\nPresupuesto mínimo para que Mask R-CNN alcance las {MASK_EPOCHS_REFERENCE} épocas de la receta original: '
          f'{recomendado} min. Fijar BUDGET_MINUTES_PER_RUN una sola vez, antes de la primera corrida completa.')


## 6. Entrenamiento equiparado por tiempo observado de sesión GPU

**Control para gastar unidades:** esta celda no ejecuta entrenamientos hasta que `RUN_FULL=True`; por defecto entrena solo la semilla de `ACTIVE_SEEDS`. Presupuesto de referencia: `BUDGET_MINUTES_PER_RUN` por arquitectura y semilla. El experimento inicial conserva datos/test, pesos de base COCO, tamaño de entrada, optimizador y congelamiento heredados. Las diferencias de receta siguen siendo una limitación; solo el **presupuesto de tiempo en L4** es el factor restringido.

**Tolerancia:** finalización al límite de épocas significa que los tiempos NO serán idénticos. Un presupuesto nominal igual no justifica afirmar costo real igual si el CSV muestra variaciones importantes. Todos los resultados se registran en una carpeta nueva y no sobrescriben NB03/NB06/NB22/NB31/NB32/NB33.


In [ ]:
def train_yolo_matched(model_name, seed):
    tag=model_name.replace('.pt','');folder=OUT/'matched'/tag/f'seed_{seed}'
    meta=folder/'finish.json'
    budget_sec=BUDGET_MINUTES_PER_RUN*60
    if meta.exists():
        info=json.loads(meta.read_text(encoding='utf-8'))
        assert info['frozen_inputs']==HASHES and info['budget_seconds_target']==budget_sec
        assert (folder/'weights'/'best.pt').is_file()
        print('Ya existe',tag,seed,': se reutiliza (sin reentrenar).')
        return info
    if folder.exists() and (folder/'weights'/'last.pt').exists():
        raise RuntimeError('Existe corrida YOLO interrumpida sin finish.json. Auditar; no reiniciar encima: '+str(folder))
    folder.mkdir(parents=True,exist_ok=True)
    torch.cuda.empty_cache();gc.collect()
    model=YOLO(model_name)
    t0=time.monotonic()
    model.train(data=str(YAML),epochs=YOLO_EPOCH_CAP,time=BUDGET_MINUTES_PER_RUN/60.0,
                imgsz=YOLO_IMGSZ,batch=YOLO_BATCH,seed=seed,deterministic=True,
                device=0,workers=2,optimizer='auto',amp=True,patience=YOLO_EPOCH_CAP,
                close_mosaic=10,plots=False,save=True,save_period=5,
                project=str(OUT/'matched'/tag),name=f'seed_{seed}',exist_ok=True)
    torch.cuda.synchronize();elapsed=time.monotonic()-t0
    best=folder/'weights'/'best.pt';history=folder/'results.csv'
    assert best.is_file() and history.is_file(), 'Faltan best.pt o results.csv de '+str(folder)
    df=pd.read_csv(history);df.columns=df.columns.str.strip()
    val_col=next((c for c in df.columns if 'mAP50-95(M)' in c),None)  # columna de AP de máscara en Ultralytics
    best_val=float(df[val_col].max()) if val_col else None
    info={'model':tag,'seed':seed,'elapsed_seconds':elapsed,'budget_seconds_target':budget_sec,
          'deviation_pct':100*(elapsed-budget_sec)/budget_sec,'epochs_finished':len(df),
          'best_validation_seg_AP':best_val,'best_weights_sha256':sha256_file(best),
          'frozen_inputs':HASHES,'train_dataset':'FaroPigSeg split_fecha_v1 (train/val only)',
          'architecture_recipe':'Ultralytics optimizer=auto; 640px; batch 8; COCO pretrained',
          'test_evaluated':False,'gpu':GPU,'finished_utc':datetime.now(timezone.utc).isoformat()}
    write_json_atomic(meta,info)
    return info


In [ ]:
def train_maskrcnn_matched(seed):
    folder=OUT/'matched'/'maskrcnn_resnet50_fpn'/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    meta=folder/'finish.json';last=folder/'maskrcnn_last.pt';best=folder/'maskrcnn_best.pt';hist=folder/'history.csv'
    budget_sec=BUDGET_MINUTES_PER_RUN*60
    if meta.exists():
        info=json.loads(meta.read_text(encoding='utf-8'))
        assert info['frozen_inputs']==HASHES and info['budget_seconds_target']==budget_sec and best.is_file()
        print('MaskRCNN',seed,'ya finalizado; no se repite.')
        return info
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    train_ds=PigMasks(cf['train']);val_ds=PigMasks(cf['val'])
    dl=DataLoader(train_ds,batch_size=MASK_BATCH,shuffle=True,num_workers=0,collate_fn=collate_fn)
    model=build_maskrcnn(pretrained=not last.exists()).to('cuda')
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    epochs=[];start_epoch=1;best_ap=-1;elapsed_prior=0.0
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        assert state['frozen_inputs']==HASHES and state['budget_seconds_target']==budget_sec and state['seed']==seed
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler']);start_epoch=state['epoch']+1
        best_ap=state['best_ap'];elapsed_prior=state['elapsed_seconds_total']
        epochs=pd.read_csv(hist).to_dict('records') if hist.exists() else []
        print('Reanudar MaskRCNN seed',seed,'desde época',start_epoch)
    launch_t0=time.monotonic();elapsed=elapsed_prior
    for epoch in range(start_epoch,MASK_EPOCH_CAP+1):
        if elapsed>=budget_sec:break
        model.train();losses=[];train_t0=time.monotonic()
        for images,targets in dl:
            images=[x.to('cuda') for x in images]
            targets=[{k:v.to('cuda') for k,v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                loss_dict=model(images,targets);loss=sum(loss_dict.values())
            if not torch.isfinite(loss): raise RuntimeError('Loss Mask R-CNN no finita; inspeccionar datos/GPU')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer);scaler.update();losses.append(float(loss.item()))
            if time.monotonic()-launch_t0+elapsed_prior>=budget_sec:break
        torch.cuda.synchronize()
        time_train=time.monotonic()-train_t0
        # Validar cada dos épocas, o en la última disponible. Nunca usar test.
        time_before_val=time.monotonic()-launch_t0+elapsed_prior
        do_val=(epoch%2==0 or time_before_val>=budget_sec or epoch==MASK_EPOCH_CAP)
        val_ap=None
        if do_val:
            val_metrics,_=coco_metric_for_model(model,val_ds,torch.device('cuda'),desc=f'Val MaskRCNN {seed}/{epoch}')
            val_ap=float(val_metrics['seg_map50_95'])
            if val_ap>best_ap+1e-4:
                best_ap=val_ap
                torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                            'epoch':epoch,'best_val_ap':best_ap,'seed':seed,'frozen_inputs':HASHES},best)
        elapsed=time.monotonic()-launch_t0+elapsed_prior
        row={'model':'maskrcnn_resnet50_fpn','seed':seed,'epoch':epoch,'train_seconds':time_train,
             'elapsed_seconds_total':elapsed,'train_loss_mean':np.mean(losses),'val_AP50_95':val_ap,
             'train_images_processed_this_epoch':len(losses)*MASK_BATCH}
        epochs.append(row);pd.DataFrame(epochs).to_csv(hist,index=False)
        state={'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
               'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
               'epoch':epoch,'seed':seed,'best_ap':best_ap,
               'elapsed_seconds_total':elapsed,'budget_seconds_target':budget_sec,'frozen_inputs':HASHES}
        torch.save(state,last)
        print('MaskRCNN',seed,'epoch',epoch,'elapsed sec',round(elapsed,1),'val_AP',val_ap)
        if elapsed>=budget_sec:break
    if not best.exists():
        # Si se interrumpió antes de la primera validación, forzar una evaluación válida.
        val_metrics,_=coco_metric_for_model(model,val_ds,torch.device('cuda'),desc='Val final')
        best_ap=float(val_metrics['seg_map50_95'])
        torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                    'epoch':epoch,'best_val_ap':best_ap,'seed':seed,'frozen_inputs':HASHES},best)
        elapsed=time.monotonic()-launch_t0+elapsed_prior
    info={'model':'maskrcnn_resnet50_fpn','seed':seed,'elapsed_seconds':elapsed,
          'budget_seconds_target':budget_sec,'deviation_pct':100*(elapsed-budget_sec)/budget_sec,
          'epochs_finished':len(epochs),'best_validation_seg_AP':best_ap,'best_weights_sha256':sha256_file(best),
          'frozen_inputs':HASHES,'train_dataset':'FaroPigSeg split_fecha_v1 (train/val only)',
          'architecture_recipe':'torchvision COCO v1; AdamW 1e-4; batch 1; 512-768px; backbone partial freeze',
          'test_evaluated':False,'gpu':GPU,'finished_utc':datetime.now(timezone.utc).isoformat()}
    write_json_atomic(meta,info)
    del model,optimizer,scaler,dl,train_ds,val_ds;gc.collect();torch.cuda.empty_cache()
    return info


## 7. Ejecutar las nueve corridas (solo cuando `RUN_FULL=True`)

In [ ]:
def _seed_done(sd):
    tags = ('yolo11n-seg', 'yolo26n-seg', 'maskrcnn_resnet50_fpn')
    return all((OUT/'matched'/t/f'seed_{sd}'/'finish.json').exists() for t in tags)
if ACTIVE_SEEDS == 'auto':
    pendientes = [s for s in SEEDS if not _seed_done(s)]
    ACTIVE_SEEDS = tuple(pendientes[:1])
    print('Semillas completas:', [s for s in SEEDS if _seed_done(s)], '| semilla de esta sesión:', ACTIVE_SEEDS or 'ninguna (las 9 corridas ya terminaron)')
assert set(ACTIVE_SEEDS).issubset(set(SEEDS)) and len(ACTIVE_SEEDS) <= 1, 'Una sola semilla por sesión.'
if RUN_FULL and ACTIVE_SEEDS:
    assert GPU_NAME_REQUIRED.lower() in torch.cuda.get_device_name(0).lower()
    all_results=[]
    # Estructura intercalada por semilla para evitar que el orden favorezca siempre una arquitectura.
    for sd in ACTIVE_SEEDS:
        for name in ('yolo11n-seg.pt','yolo26n-seg.pt'):
            print('ENTRENAMIENTO',name,'seed',sd)
            all_results.append(train_yolo_matched(name,sd))
        print('ENTRENAMIENTO MaskRCNN seed',sd)
        all_results.append(train_maskrcnn_matched(sd))
        pd.DataFrame(all_results).to_csv(OUT/'matched_budget_summary.csv',index=False)
    print('✅ Semillas ejecutadas:',ACTIVE_SEEDS,'; resultados parciales 3/9, 6/9 o 9/9. No se evaluó test.')
elif RUN_FULL:
    print('✅ Las 9 corridas ya están completas; no se entrena nada más.')
else:
    print('✅ Entrenamiento completo NO ejecutado; para activarlo, poner RUN_FULL=True en la celda 1 y repetir todas las celdas.')


## 8. Auditoría y figuras de validación (no prueba)

In [ ]:
records=[]
for meta in sorted((OUT/'matched').rglob('finish.json')) if (OUT/'matched').is_dir() else []:
    d=json.loads(meta.read_text(encoding='utf-8'))
    assert d['frozen_inputs']==HASHES and not d['test_evaluated']
    if d.get('best_validation_seg_AP') is None and d['model'].startswith('yolo'):
        # versiones anteriores dejaron el AP de YOLO vacío: se recupera de results.csv sin modificar finish.json
        h = meta.parent/'results.csv'
        if h.is_file():
            t = pd.read_csv(h); t.columns = t.columns.str.strip()
            c = next((c for c in t.columns if 'mAP50-95(M)' in c), None)
            if c: d['best_validation_seg_AP'] = float(t[c].max()); d['best_ap_source'] = 'results.csv'
    records.append(d)
if records:
    df=pd.DataFrame(records).sort_values(['seed','model'])
    df.to_csv(OUT/'matched_budget_summary.csv',index=False)
    display(df[['model','seed','budget_seconds_target','elapsed_seconds','deviation_pct','epochs_finished','best_validation_seg_AP']])
    n=len(df);print('Corridas encontradas:',n,'/9', '| semillas completas:', sorted(df.groupby('seed').size().loc[lambda x: x==3].index.tolist()))
    tolerance_ok=df['deviation_pct'].abs().le(100*RELATIVE_WALL_TIME_TOLERANCE)
    df['within_walltime_tolerance']=tolerance_ok
    df.to_csv(OUT/'matched_budget_summary.csv',index=False)
    if not bool(tolerance_ok.all()):print('⚠️ Hay tiempos reales fuera del ±10%. No presentar esto como presupuesto efectivamente igualado sin revisión.')
    plt.figure(figsize=(8,4));
    for model,group in df.groupby('model'):
        plt.scatter([model]*len(group),group['elapsed_seconds']/60,label=model)
    plt.axhline(BUDGET_MINUTES_PER_RUN,color='black',linestyle='--',linewidth=1,label='objetivo')
    plt.ylabel('Tiempo de sesión GPU observado (min)');plt.xticks(rotation=10);plt.legend();plt.tight_layout()
    plt.savefig(OUT/'matched_budget_actual_minutes.png',dpi=180);plt.show()
    print('✅ Informe:',OUT/'matched_budget_summary.csv')
else:print('No hay entrenamiento completo todavía. Solo se reporta piloto.')


## 9. Decisión científica y próximos pasos

**Antes de escribir el nuevo manuscrito:**
- Analizar el CSV de tiempos. Si tres arquitecturas se alejaron más del **±10% de tiempo**, declarar *budget-capped* en lugar de *matched-compute*, o ajustar solo un futuro experimento nuevo con presupuesto corregido.
- Separar **tiempo entrenamiento/validación** de la **eficiencia inferencial**. Una hora L4 no implica FLOPs, energía o imágenes procesadas equivalentes.
- Guardar los nueve checkpoints `best` y los manifiestos del protocolo. No modificar ni reemplazar los resultados históricos (NB22, NB31, NB32 y NB33).
- Para evaluación final, reutilizar el **evaluador COCO común** de NB33 sobre los 373 test existentes y no optimizar hiperparámetros, checkpoints o umbrales con test.
- El análisis posterior será **retrospectivo y no ciego**, porque el conjunto test ya se inspeccionó durante el desarrollo. No alegar validación independiente entre granjas ni independencia de cámaras, animales o corrales.

**Archivos esperados:** `pilot_summary.csv`, `protocol_config_v1.json`, `matched_budget_summary.csv`, `matched_budget_actual_minutes.png`, carpetas por arquitectura/semilla con `best` y metadatos hash. La carpeta de salida es `03_Resultados/54_Presupuesto_Equiparado_L4`.
